In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
den_reviews = pd.read_parquet("../data/denver_restaurant_reviews.parquet")
den_restaurants = pd.read_parquet("../data/denver_restaurants.parquet")

In [4]:
den_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 1203229 entries, 0 to 1203228
Data columns (total 11 columns):
 #   Column           Non-Null Count    Dtype  
---  ------           --------------    -----  
 0   gmap_id          1203229 non-null  str    
 1   business_name    1203229 non-null  str    
 2   address          1202906 non-null  str    
 3   categories       1203229 non-null  str    
 4   latitude         1203229 non-null  float64
 5   longitude        1203229 non-null  float64
 6   business_rating  1203229 non-null  float64
 7   user_id          1203229 non-null  str    
 8   rating           1203229 non-null  int64  
 9   review_text      1203229 non-null  str    
 10  review_time      1203229 non-null  int64  
dtypes: float64(3), int64(2), str(6)
memory usage: 478.5 MB


In [8]:
# Model
import joblib

model = joblib.load("../models/sentiment_logistic_model.joblib")
tfidf = joblib.load("../models/tfidf_vectorizer.joblib")



/opt/anaconda3/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/opt/anaconda3/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.2 when using version 1.9.0. This might lead t

In [34]:
sample = [den_reviews["review_text"][0]]

vector_sample = tfidf.transform(sample)
pred = model.predict(vector_sample)

print(sample)
print(pred)

['It is an authentic Arabic taste. I have tried different dishes. Recommend lamb shank, grilled fish, fried fish, chicken shawarma and chicken  kabab.']
[1]


In [57]:
df = pd.concat([den_reviews["gmap_id"][:5],output_df],axis=1)
df

,gmap_id,sentiment_score
0,0x876c7d7fca755a37:0xd6ed2c0ad5485823,1
1,0x876c7d7fca755a37:0xd6ed2c0ad5485823,1
2,0x876c7d7fca755a37:0xd6ed2c0ad5485823,0
3,0x876c7d7fca755a37:0xd6ed2c0ad5485823,0
4,0x876c7d7fca755a37:0xd6ed2c0ad5485823,1


In [61]:
reduced_den_reviews = den_reviews[["gmap_id", "business_name", "address","categories","business_rating", "user_id","review_text"]]

In [62]:
reduced_den_reviews

,gmap_id,business_name,address,categories,business_rating,user_id,review_text
0,0x876c7d7fca755a37:0xd6ed2c0ad5485823,Ali Grill,"Ali Grill, 1155 S Havana St unit 35, Aurora, C...",Middle Eastern restaurant,4.5,110367972631566169908,It is an authentic Arabic taste. I have tried ...
1,0x876c7d7fca755a37:0xd6ed2c0ad5485823,Ali Grill,"Ali Grill, 1155 S Havana St unit 35, Aurora, C...",Middle Eastern restaurant,4.5,103100164339259283824,Awesome place the fish is excellent. You can g...
2,0x876c7d7fca755a37:0xd6ed2c0ad5485823,Ali Grill,"Ali Grill, 1155 S Havana St unit 35, Aurora, C...",Middle Eastern restaurant,4.5,115805455801223941952,Drove half an hour to try this place out becau...
3,0x876c7d7fca755a37:0xd6ed2c0ad5485823,Ali Grill,"Ali Grill, 1155 S Havana St unit 35, Aurora, C...",Middle Eastern restaurant,4.5,113322511197892113942,There is a lack of product knowledge and a lac...
4,0x876c7d7fca755a37:0xd6ed2c0ad5485823,Ali Grill,"Ali Grill, 1155 S Havana St unit 35, Aurora, C...",Middle Eastern restaurant,4.5,117880162302982807571,Chicken Shawarmas were very good. The hummus a...
...,...,...,...,...,...,...,...
1203224,0x876c8676ce47e5eb:0x524c2f26efcd075b,Surena Persian Cuisine,"Surena Persian Cuisine, 9625 E Arapahoe Rd P, ...","Mediterranean restaurant, Fine dining restaura...",4.6,108033996086673186798,(Translated by Google) Very rich everything\n\...
1203225,0x876c8676ce47e5eb:0x524c2f26efcd075b,Surena Persian Cuisine,"Surena Persian Cuisine, 9625 E Arapahoe Rd P, ...","Mediterranean restaurant, Fine dining restaura...",4.6,108055606839040459539,(Translated by Google) Super food\nVery tasty ...
1203226,0x876c8676ce47e5eb:0x524c2f26efcd075b,Surena Persian Cuisine,"Surena Persian Cuisine, 9625 E Arapahoe Rd P, ...","Mediterranean restaurant, Fine dining restaura...",4.6,105126483929402927557,(Translated by Google) A good restaurant\n\n(O...
1203227,0x876c8676ce47e5eb:0x524c2f26efcd075b,Surena Persian Cuisine,"Surena Persian Cuisine, 9625 E Arapahoe Rd P, ...","Mediterranean restaurant, Fine dining restaura...",4.6,105048750403986990757,(Translated by Google) Best Kabab in Colorado!...


In [71]:
score = sentiment_score(den_reviews)

score_df = pd.DataFrame({"sentiment_score": score[0]})
df = pd.concat([reduced_den_reviews, score_df], axis = 1)

In [75]:
mean_score_agg = df.groupby('gmap_id')['sentiment_score'].mean()

In [138]:
def sentiment_compute(df, model, vectorizer):

    df = df.copy()
    cols = ['gmap_id', 'business_name', 'address','categories', 'business_rating', 'business_review_count']

    vectorized_reviews = vectorizer.transform([i for i in df["review_text"]])
    df["sentiment_score"] = model.predict(vectorized_reviews)

    avg_score = df.groupby('gmap_id', as_index= False)['sentiment_score'].mean()

    business_df = den_restaurants[cols].drop_duplicates(subset = "gmap_id")

    final_df = pd.merge(business_df, avg_score, on="gmap_id", how = "inner")

    return final_df


In [139]:
den_res_15 = den_restaurants[den_restaurants["business_review_count"] >= 50]
merge_revres = pd.merge(den_res_15, den_reviews, on="gmap_id", how= "inner")

In [141]:
output = sentiment_compute(merge_revres, model, tfidf)

In [142]:
output

,gmap_id,business_name,address,categories,business_rating,business_review_count,sentiment_score
0,0x876c7f3d6abe7f95:0x97ef764a117ba416,Deviation Distilling,"Deviation Distilling, 1821 Blake St #130, Denv...","Distillery, Small plates restaurant",5.0,58,1.000000
1,0x876c830f3242a585:0xbf314a775cdf84df,Wing Wok,"Wing Wok, 7530 S University Blvd, Littleton, C...",Korean restaurant,4.3,105,0.781250
2,0x876c7f16d35d7343:0xa4c93839c7da47c,Ti Cafe,"Ti Cafe, 30 N Broadway, Denver, CO 80203",Coffee shop,4.9,58,1.000000
3,0x876c8f2ee1688e71:0x2b4b8fd3e157d93,ProteinHouse,"ProteinHouse, 6790 S Cornerstar Way unit a, Au...",Restaurant,3.6,55,0.750000
4,0x876b812a17208475:0x2d7584943ef55383,Subway,"Subway, 440 Wadsworth Blvd, Lakewood, CO 80226","Sandwich shop, Caterer, Fast food restaurant, ...",2.5,78,0.242424
...,...,...,...,...,...,...,...
5546,0x876c7e9685f7635b:0xd459c43891e85c1b,North Italia,"North Italia, 190 Clayton Ln, Denver, CO 80206","Italian restaurant, Takeout Restaurant, Pizza ...",4.6,1124,0.892135
5547,0x876c8677cfddf799:0xe88d7bd5d8e7a63a,New Indian Cafe,"New Indian Cafe, 6886 S Clinton St, Greenwood ...","Indian restaurant, Bar, Biryani restaurant, Bu...",4.1,698,0.790393
5548,0x876c78d42afbf349:0x2dc438f7427d01b7,Noodles and Company,"Noodles and Company, 303 16th St Mall Suite 15...","Restaurant, Down home cooking restaurant, Take...",3.8,478,0.662577
5549,0x876c78c4fd3b211b:0x41934a9b627431bb,P.F. Chang's,"P.F. Chang's, 1415 15th St, Denver, CO 80202","Chinese restaurant, Asian restaurant, Dim sum ...",4.1,934,0.707042


In [143]:
output.to_csv("business_sentiment_score")